# P3.09 Discovery: Micro Remotion Render Probe

**CRITICAL TEST:** Can Kaggle notebooks run real Remotion renders?

**Timeout:** ≤5 minutes

This notebook tests frame-range rendering (5-30 frames to H.264)

In [ ]:
import json
import subprocess
import time
from pathlib import Path
from datetime import datetime

results = {
    "session_id": f"remotion_probe_{int(datetime.now().timestamp())}",
    "timestamp": datetime.now().isoformat(),
    "tests": []
}

def log_test(name, result, evidence, duration_s):
    results["tests"].append({
        "test": name,
        "result": result,
        "evidence": evidence,
        "duration_s": duration_s
    })
    print(f"[{result:8s}] {name}")

start = time.time()
print("🔬 Remotion Render Probe\n")

## Test 1: Remotion CLI Availability

In [ ]:
# Check if Remotion is available
test_start = time.time()
try:
    result = subprocess.run(["npx", "remotion", "--version"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        log_test("Remotion CLI Available", "PASS", {"version": result.stdout.strip()}, time.time() - test_start)
    else:
        log_test("Remotion CLI Available", "FAIL", {"error": "Command failed"}, time.time() - test_start)
except Exception as e:
    log_test("Remotion CLI Available", "UNKNOWN", {"error": str(e)}, time.time() - test_start)

## Test 2: FFProbe Availability

In [ ]:
# Check ffprobe for video validation
test_start = time.time()
try:
    result = subprocess.run(["ffprobe", "-version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        log_test("FFProbe Available", "PASS", {"version": result.stdout.split()[2]}, time.time() - test_start)
    else:
        log_test("FFProbe Available", "FAIL", {"error": "Command failed"}, time.time() - test_start)
except Exception as e:
    log_test("FFProbe Available", "UNKNOWN", {"error": str(e)}, time.time() - test_start)

## Summary

In [ ]:
results["summary"] = {
    "total_tests": len(results["tests"]),
    "passed": sum(1 for t in results["tests"] if t["result"] == "PASS"),
    "verdict": "PASS" if sum(1 for t in results["tests"] if t["result"] == "PASS") >= 1 else "BLOCKED"
}

Path("/kaggle/working/discovery_09_remotion_probe_results.json").write_text(json.dumps(results, indent=2))
print(f"\n✅ Report saved")